# Multi-Platform Content Catalog Advanced Analytics & ML Pipeline

This notebook demonstrates the end-to-end data analytics and machine learning pipeline for our unified streaming platform catalog containing titles from **Netflix**, **Disney+**, and **Amazon Prime Video**:
1. **Unified Schema Alignment** — Standardizing raw metadata across Netflix, Disney+, and Amazon Prime.
2. **Tableau-style Interactive Visualizations** — Drag-and-drop analytics utilizing **PyGWalker**.
3. **Comparative Exploratory Data Analysis** — Platform content counts, type distributions, and genre breakdowns.
4. **K-Means Clustering** — Segmenting unified content profiles based on ratings and popularity.
5. **Naive Bayes Quality Classifier** — Training a supervised model on rated titles to predict catalog quality probabilities.
6. **Association Rule Mining** — Mining frequent co-occurrences of genres to uncover global catalog bundling patterns.
7. **Time-Series Release Forecasting** — Modeling and projecting catalog expansions for each platform over the next 10 years.

### Step 1: Import Libraries and Load Unified Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting styles
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Load the unified dataset created by backend/ingest.py
df = pd.read_csv("../datasets/cleaned_dataset.csv")
print(f"Unified catalog contains {df.shape[0]} titles across {df['platform'].nunique()} platforms.")
df.head()

### Step 2: Comparative Platform EDA

In [ ]:
# 1. Catalog Size Comparison
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='platform', palette='Set1', hue='platform', legend=False)
plt.title('Total Catalog Library Size comparison')
plt.xlabel('Streaming Platform')
plt.ylabel('Number of Titles')
plt.show()

In [ ]:
# 2. Movie vs TV Show Distribution per Platform
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='platform', hue='type', palette='muted')
plt.title('Movies vs TV Shows Distribution across Platforms')
plt.xlabel('Platform')
plt.ylabel('Count')
plt.legend(title='Content Type')
plt.show()

### Step 3: Interactive Visualizations using PyGWalker
Following the **Kanaries PyGWalker Netflix Data** exploration methodology, we instantiate a Tableau-like drag-and-drop workspace directly in this Jupyter Notebook to easily filter, explore, and chart the unified cross-platform catalog.

In [ ]:
import pygwalker as pyg

# Launch pygwalker interactive workspace over combined Netflix, Disney+, and Amazon Prime datasets
walker = pyg.walk(df)

### Step 4: K-Means Clustering on content ratings & popularity

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Cluster all titles based on rating and popularity (filtering to Netflix rated titles for visual dispersion)
rated_df = df[df['vote_count'] > 0].copy()
features = ['popularity', 'vote_average']

scaler = StandardScaler()
df_scaled = scaler.fit_transform(rated_df[features].fillna(0))

# Fit K-Means (K=4)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rated_df['cluster'] = kmeans.fit_predict(df_scaled)

# Scatter plot
plt.figure(figsize=(10, 8))
sns.scatterplot(data=rated_df, x='vote_average', y='popularity', hue='cluster', palette='Set1', alpha=0.7)
plt.yscale('log')
plt.title('Movie Catalog Segmentation via K-Means (Rated Catalog Subset)')
plt.xlabel('Rating (Vote Average)')
plt.ylabel('Popularity (Log Scale)')
plt.legend(title='Cluster ID')
plt.show()

### Step 5: Naive Bayes Quality Classification
We train a Gaussian Naive Bayes classifier to predict whether a movie is a "Hit" (Rating >= 7.0 & Count >= 100) based on popularity and genre dummies.

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Target variable
df_rated['is_hit'] = ((df_rated['vote_average'] >= 7.0) & (df_rated['vote_count'] >= 100)).astype(int)

# Feature variables: genres encoded
genre_dummies = pd.get_dummies(df_rated['primary_genre'], prefix='genre')
X = pd.concat([df_rated[['popularity']], genre_dummies], axis=1)
y = df_rated['is_hit']

# Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit Gaussian Naive Bayes
nb = GaussianNB()
nb.fit(X_train, y_train)
y_pred = nb.predict(X_test)

print(f"Naive Bayes Quality Classifier Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred))

### Step 6: Genre Association Rule Mining
We calculate frequent co-occurrences of genres to uncover global catalog bundling patterns.

In [ ]:
# Get all unique genres
all_genres = set()
for g_list in df['clean_genres'].dropna().str.split(','):
    for g in g_list:
        all_genres.add(g.strip())
        
all_genres = sorted(list(all_genres))

# Calculate Support, Confidence, and Lift
n_samples = len(df)
rules = []

# Create binary flags
for genre in all_genres:
    df[f'genre_{genre}'] = df['clean_genres'].apply(lambda x: 1 if isinstance(x, str) and genre in x else 0)
    
supports = {genre: df[f'genre_{genre}'].sum() / n_samples for genre in all_genres}

for i in range(len(all_genres)):
    g_a = all_genres[i]
    support_a = supports[g_a]
    for j in range(len(all_genres)):
        if i == j: continue
        g_b = all_genres[j]
        support_b = supports[g_b]
        
        support_ab = ((df[f'genre_{g_a}'] == 1) & (df[f'genre_{g_b}'] == 1)).sum() / n_samples
        
        if support_ab >= 0.01:
            confidence = support_ab / support_a
            lift = confidence / support_b
            rules.append({
                "antecedent": g_a,
                "consequent": g_b,
                "support": support_ab,
                "confidence": confidence,
                "lift": lift
            })

rules_df = pd.DataFrame(rules).sort_values(by='lift', ascending=False)
rules_df.head(10)

### Step 7: Content Expansion Forecasting
We model and predict movie catalog growth trends using historical data (1990-2022) with a Linear Trend model.

In [ ]:
from sklearn.linear_model import LinearRegression

# Extract release counts per year
year_counts = df[(df['release_year'] >= 1990) & (df['release_year'] <= 2022)]['release_year'].value_counts().sort_index()
X_reg = year_counts.index.values.reshape(-1, 1)
y_reg = year_counts.values

# Fit Linear Regression
model = LinearRegression()
model.fit(X_reg, y_reg)

# Forecast for next 10 years (2023 - 2032)
future_years = np.array(range(2023, 2033)).reshape(-1, 1)
predictions = model.predict(future_years)

# Plot actual vs forecast
plt.figure(figsize=(12, 6))
plt.plot(year_counts.index, year_counts.values, marker='o', label='Actual Titles Count', color='crimson')
plt.plot(np.vstack([X_reg, future_years]).flatten(), 
         np.concatenate([model.predict(X_reg), predictions]), 
         linestyle='--', color='darkgrey', label='Linear Trend Forecast')
plt.axvline(x=2022.5, color='gold', linestyle=':', label='Forecast Start')
plt.title('Streaming Catalog Growth & 10-Year Linear Forecast')
plt.xlabel('Release Year')
plt.ylabel('Number of Titles')
plt.legend()
plt.show()